# Cold start and traffic allocation
## Goal
Turn concepts into inspectable inputs, operations, gradients, and comparisons.

Evidence: **real attributes + simulated traffic**. Local teaching parameters are not production defaults; mechanism experiments are not full reproductions of the original papers.

> **English companion.** All instructional Markdown is translated. Experiment code, Chinese language examples, execution counts, and saved outputs remain as in the source edition; historical output text may therefore be Chinese. The initial setup cell anchors the working directory and Python import path for this companion's deeper folder; it does not change the experiment. Saved outputs are historical evidence, not a new run. Run from the top after preparing the required data and models.

In [ ]:
# English companions live one directory deeper; keep the original experiment code unchanged.
from pathlib import Path as _TranslationPath
import os as _translation_os
import sys as _translation_sys
_translation_root = next((p for p in [_TranslationPath.cwd(), *_TranslationPath.cwd().parents]
                          if (p / "reclab").is_dir()), None)
if _translation_root is None:
    raise RuntimeError("Open this notebook from inside the Search Recommendation Ads Lab checkout.")
_translation_os.chdir(_translation_root)
if str(_translation_root) not in _translation_sys.path:
    _translation_sys.path.insert(0, str(_translation_root))


## Setup
On the first pass, run from Setup in order: later cells reuse earlier variables. Then change one parameter and rerun that cell and its downstream cells. Constructed data does not establish real business impact.

In [1]:
from pathlib import Path
import sys
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Code

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "reclab/models.py").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from reclab.data import Dataset
from reclab.paths import ARTIFACTS, PROCESSED, read_json

plt.rcParams.update({"figure.figsize": (9, 3.8), "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelcolor": "#182b2b", "text.color": "#182b2b",
    "axes.facecolor": "white", "figure.facecolor": "white",
    "grid.color": "#d4ddda", "axes.axisbelow": True})
dataset = Dataset()
print("Project:", PROJECT)
print("Dataset version:", dataset.manifest["dataset_version"])
print("Python:", sys.executable)
assert Path(sys.executable).parent == PROJECT / ".venv/bin", "请选择 Recommendation Lab 内核"

Project: /Users/lizuolin/local/recommendation-lab
Dataset version: 2772cab0aca8cb81
Python: /Users/lizuolin/local/recommendation-lab/.venv/bin/python


## Steps
Lecture references: 07_ColdStart_01.pdf, 07_ColdStart_02.pdf, 07_ColdStart_03.pdf, 07_ColdStart_04.pdf, 07_ColdStart_05.pdf, 07_ColdStart_06.pdf

### Category-only retrieval after hiding IDs
A stress exercise hides IDs inside the training catalog; it does not measure genuinely new products. It does not use those hidden items' trained ID vectors to claim cold-start success.

**Predict first:** What happens to this channel when category is missing or every value is unknown?

In [2]:
categories=sorted({x["category"] for x in dataset.items});lookup={c:i for i,c in enumerate(categories)}
side=np.eye(len(categories),dtype=np.float32)[[lookup[x["category"]] for x in dataset.items]]
user=next(iter(dataset.histories));history=dataset.histories[user]
profile=side[np.array(history[-20:])-1].mean(0);scores=side@profile
hidden_ids=set(range(200))
chosen=[i for i in np.argsort(-scores,kind="stable") if i in hidden_ids][:10]
display(pd.DataFrame([{"item":dataset.items[i]["item_id"],"category":dataset.items[i]["category"],"score":scores[i]} for i in chosen]))

,item,category,score
0,10034,1219,0.916667
1,18519,1219,0.916667
2,403,unknown,0.083333
3,546,unknown,0.083333
4,2008,unknown,0.083333
5,8497,unknown,0.083333
6,496,707,0.000000
7,829,707,0.000000
8,869,1650,0.000000
9,1152,1051,0.000000


### Clustering and look-alike audience expansion
A mechanism experiment using real categories and training histories. No real image/text or creator fields are available, so CNN/BERT content-similarity training is not reproduced.

**Predict first:** If the seed audience is biased, does expansion correct or amplify that bias?

In [3]:
from reclab.learning.retrieval import kmeans
centers,labels=kmeans(side[:600],clusters=8,iterations=5)
user_ids=list(dataset.histories)[:500]
profiles=np.array([side[np.array(dataset.histories[u][-20:])-1].mean(0) for u in user_ids])
seed_category=dataset.items[0]["category"]
seeds=[i for i,u in enumerate(user_ids) if any(dataset.items[j-1]["category"]==seed_category for j in dataset.histories[u])] or [0]
audience=profiles[seeds].mean(0)
audience=audience/max(float(np.linalg.norm(audience)),1e-8)
normalized=profiles/np.maximum(np.linalg.norm(profiles,axis=1,keepdims=True),1e-8)
expanded=np.argsort(-(normalized@audience))[:10]
display(pd.DataFrame({"user":[dataset.users[user_ids[i]] for i in expanded],"similarity":(normalized@audience)[expanded]}))
print("Cluster counts:",np.unique(labels,return_counts=True))

,user,similarity
0,104555,0.866971
1,113172,0.866971
2,99319,0.866382
3,77609,0.843095
4,78724,0.740322
5,19967,0.733718
6,102661,0.711574
7,44071,0.699376
8,71099,0.684696
9,39027,0.657632


Cluster counts: (array([0, 1, 2, 3, 4, 5, 6, 7]), array([557,  16,   2,   4,   4,   6,   7,   4]))


### Guaranteed volume, different objectives, and supply constraints
All traffic is simulated. The initial 100 impressions are only a lecture example. Insufficient supply makes universal guarantees impossible; even sufficient traffic does not guarantee a fixed boost will work.

**Predict first:** Can guaranteed volume, content quality, and consumer experience all be maximized simultaneously?

In [4]:
from reclab.learning.systems import allocate_quotas,bucket
quality=np.linspace(.1,1.,20);QUOTA=100;rows=[]
for requests in [500,2000,4000]:
    exposure=allocate_quotas(quality,requests,quota=QUOTA,boost=3.)
    rows.append({"requests":requests,"required_supply":len(quality)*QUOTA,
        "items_reaching_quota":int((exposure>=QUOTA).sum()),"allocated":int(exposure.sum())})
    assert exposure.sum()==requests
display(pd.DataFrame(rows))
assert rows[0]["items_reaching_quota"]<len(quality)

,requests,required_supply,items_reaching_quota,allocated
0,500,2000,0,500
1,2000,2000,1,2000
2,4000,2000,1,4000


### Traffic interference in creator-side A/B testing
Construct two creator pools. Under a shared budget, additional traffic to A displaces B. No causal conclusion about real creator posting or user retention follows.

**Predict first:** How can you distinguish net business growth from traffic transferred between experiment groups?

In [5]:
budget=10000;base=np.array([1.,1.]);boost=np.array([1.5,1.])
before=budget*base/base.sum();after=budget*boost/boost.sum()
display(pd.DataFrame({"before":before,"after":after,"delta":after-before},index=["authorA","authorB"]))
assert np.isclose((after-before).sum(),0)
print("Reproducible author buckets:",[bucket(i,"author",2) for i in range(6)])

,before,after,delta
authorA,5000.0,6000.0,1000.0
authorB,5000.0,4000.0,-1000.0


Reproducible author buckets: [0, 1, 1, 1, 0, 0]


## Checks
Every assertion and output is traceable to code. They check operators, data snapshots, or mechanisms, not online gains.

## Next steps
Record your prediction, change only one variable, and run the relevant experiment cell. Open `reclab/learning/` to inspect helper functions.